# Database generation for the swe 1D case

This notebook relies on the swe module downloadable at https://github.com/cerea-daml/adomo-mocis-wape-daml-tutorial/blob/master/shallow_water_model.py

## Generation function

In [ ]:
# import standard modules
import os
from typing import List
from tqdm import tqdm
import numpy as np
from scipy.stats.qmc import Halton
from matplotlib import pyplot as plt
from matplotlib import animation
import seaborn as sns
from tqdm.auto import trange
from IPython.display import HTML

# for plot customisation
sns.set_context('notebook')
sns.set_style('darkgrid')
plt.rc('axes', linewidth=1)
plt.rc('axes', edgecolor='k')
plt.rc('figure', dpi=60)
palette = sns.color_palette('deep')

# import custom shallow water model package
import shallow_water_model as swm

In [2]:
def initialise_state(sw_model:swm.ShallowWaterModel,
                     heights: List,
                     widths: List,
                     positions: List,
                     ):
    r"""Create a new model state.

    The water height is a sum of crenels, each defined by a width, a height and a position
    heights, widths, positions: list of height, width and position of each crenel
    """
    state = swm.ShallowWaterState(sw_model.Nx)
    for h,w,x in zip(heights, widths, positions):
        i_min = int( x - w / 2 / sw_model.dx)
        i_max = int( x + w / 2 / sw_model.dx)
        state.h[i_min:i_max+1] += h
    return state

In [3]:
# create a driver
def forecast_driver(model, state, Nt):
    """Run a simulation of Nt time steps of the given model starting from state.
    
    Return the history of h and u.
    """
    # allocate memory
    hist = dict(Nt=Nt,
                h=np.empty((Nt+1, model.Nx)),
                u=np.empty((Nt+1, model.Nx)))
    
    # initialisation
    hist['h'][0] = state.h
    hist['u'][0] = state.u
    
    # run the Nt time steps
    for t in range(Nt):
        model.forward(state)
        hist['h'][t+1] = state.h
        hist['u'][t+1] = state.u
        
    # return history
    return hist

In [4]:
def save_simu(save_path, name, resu):
    for k,v in resu.items():
        np.save(save_path+ os.sep + f'{name}_{k}.npy', v)

def generate_dataset(
        sw_model: swm.ShallowWaterModel,
        n_waves: int,
        n_simulations: int,
        save_path: str,
        save_name:str,
        width_range = [4, 15],
        height_range = [0.02, 0.08],
        save_freqs = 10,
        seed = None
    ):

    domain_width = sw_model.dx * sw_model.Nx

    # We use QMC to sample parameters
    # In addition to better explore the parameter space
    # This reduce overlap in inital wave placement
    # especially for large cases simulation

    # gen = np.random.default_rng(seed)
    # positions, widths, heights = gen.random((3,n_simulations, n_waves))
    generator = Halton(d = 3 * n_waves, seed=seed)
    positions, widths, heights = generator.random(n_simulations).reshape(n_simulations, n_waves,3).T

    positions = positions.T * domain_width
    widths = width_range[0] + widths.T * (width_range[1] - width_range[0])
    heights = height_range[0] + heights.T * (height_range[1] - height_range[0])

    all_h = []
    all_u = []

    # Run simulations
    for idx in tqdm(range(n_simulations)):
        state = initialise_state(sw_model, heights[idx], widths[idx], positions[idx])
        hist = forecast_driver(sw_model, state, Nt=500)
        all_h.append(hist['h'][::save_freqs])
        all_u.append(hist['u'][::save_freqs])

    all_h = np.stack(all_h, axis = 0)
    all_u = np.stack(all_u, axis = 0)
    
    #save
    os.makedirs(save_path, exist_ok=True)
    np.save(save_path + os.sep + f'{save_name}_pos.npy', np.arange(sw_model.Nx) * sw_model.dx)
    np.save(save_path + os.sep + f'{save_name}_h.npy', all_h)
    np.save(save_path + os.sep + f'{save_name}_u.npy', all_u)

## Small scale data: 1k simulations of 3 waves within [0,100]

In [ ]:
Nx = 101
dx = 1
seed = 0

sw_model = swm.ShallowWaterModel(Nx, dx, Q = 0)

generate_dataset(sw_model, n_waves=3, n_simulations=1000, save_path='../data/academic_datasets/swe_1D', save_name='s1', seed = seed)

100%|██████████| 1000/1000 [00:01<00:00, 590.55it/s]


In [ ]:
#visualise an random simulation
idx = 33
h = np.load('../data/academic_datasets/swe_1D/s1_h.npy')[idx]

anim = swm.make_fancy_animation_h([('Truth', h)],
                                  'Truth simulation',
                                  y_max=1.12,
                                  freq = 1,
                                  Nt = 50)

fig = anim._fig          
ax = fig.axes[0]

anim._blit = False

HTML(anim.to_jshtml())

# Larger scales: 50 simulation per scale for a range of scale

In [ ]:
for scale in [2, 3, 7, 10]:
    print(f'Generation for scale={scale}')
    Nx = 101 * scale
    dx = 1
    seed = scale
    n_waves = 3 * scale

    sw_model = swm.ShallowWaterModel(Nx, dx, Q = 0)

    generate_dataset(sw_model, n_waves=n_waves, n_simulations=50, save_path='../data/academic_datasets/swe_1D', save_name=f's{scale}', seed = seed)

Generation for scale=2


100%|██████████| 50/50 [00:00<00:00, 360.30it/s]


Generation for scale=3


100%|██████████| 50/50 [00:00<00:00, 318.47it/s]


Generation for scale=7


100%|██████████| 50/50 [00:00<00:00, 159.06it/s]


Generation for scale=10


100%|██████████| 50/50 [00:00<00:00, 113.99it/s]


In [ ]:
#visualise an random simulation
idx = 33
scale = 2
h = np.load(f'../data/academic_datasets/swe_1D/s{scale}_h.npy')[idx]

anim = swm.make_fancy_animation_h([('Truth', h)],
                                  'Truth simulation',
                                  y_max=1.12,
                                  x_max = 100 * scale,
                                  Nx = 101 * scale - 1,
                                  freq = 1,
                                  Nt = 50)

fig = anim._fig          
ax = fig.axes[0]        

anim._blit = False

HTML(anim.to_jshtml())